# DESCARGA LIDAR

In [1]:
import geopandas as gpd

municipios = gpd.read_file('BRUTOS/CONCEJOS/DIADMI_Pol_Concejo.shp')
municipio = municipios[municipios.CONCEJO == 'Rocaforte']

In [2]:
municipio_name = 'ROCAFORTE'

In [3]:
import os

os.makedirs(f'BRUTOS/LIDAR/{municipio_name}', exist_ok=True)

In [4]:
cuad = gpd.read_file('BRUTOS/LIDAR/CUADRICULAS/lida3_navarra.gpkg')

In [5]:
lidar = cuad[cuad.intersects(municipio.iloc[0].geometry)]

In [10]:
cod_serie = 'LIDA3'
cod_prov=31

In [18]:
import os
import time
import requests
from IPython.display import display
from ipywidgets import IntProgress
from requests.adapters import HTTPAdapter
from urllib3.util import Retry

# Endpoints oficiales del flujo combinado
url_buscar_serie = "https://centrodedescargas.cnig.es/CentroDescargas/archivosSerie"
url_descarga = "https://centrodedescargas.cnig.es/CentroDescargas/descargaDir"

# 1. Configurar la sesión y la estrategia de reintentos robusta que ya te funciona
session = requests.Session()

retry = Retry(
    total=5,
    connect=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
)

adapter = HTTPAdapter(max_retries=retry)
session.mount("http://", adapter)
session.mount("https://", adapter)

# Inyección crítica de cabeceras humanas
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "es-ES,es;q=0.9,en;q=0.8",
})

# 2. TU IDEA: Simular la búsqueda previa para registrar la sesión en el servidor
# Asegúrate de tener definidas las variables 'cod_serie' y 'cod_prov' en tu entorno
payload_busqueda = {
    "codSerie": cod_serie,
    "codAgr": "MOMDT",
    "codTipoArchivo": "LAZ",
    "codProvincia": cod_prov,
    "numPagina": 1,
}

try:
    print("🔑 Registrando sesión legítima en el servidor mediante simulación de búsqueda...")
    r_busqueda = session.post(url=url_buscar_serie, data=payload_busqueda, timeout=20)
    r_busqueda.raise_for_status()
    print("✅ Sesión registrada correctamente en el CNIG. Iniciando descargas del GeoDataFrame...\n")
    
    # 3. Configurar e iniciar la barra de progreso para tus archivos
    total_ficheros = len(lidar)
    progreso = IntProgress(min=0, max=total_ficheros, description="Descargando:")
    display(progreso)

    # 4. Bucle de descarga utilizando la sesión ya validada
    for posicion, (i, row) in enumerate(lidar.iterrows(), start=1):
        nombre_fichero = str(row["nombre"]).strip()
        id_secuencial = str(row["num"]).strip()

        print(f"[{posicion}/{total_ficheros}] Descargando: {nombre_fichero} (ID: {id_secuencial})")

        payload_descarga = {
            "secuencialDescDir": '12339635',
            "nav_signature": "HUMAN_CONFIRMED",
        }

        try:
            # Petición POST de descarga con stream=True dentro de la misma sesión autorizada
            with session.post(url=url_descarga, data=payload_descarga, stream=True, timeout=45) as r_file:
                r_file.raise_for_status()

                # Control crítico: Si el cortafuegos nos rechaza o expira la sesión, devolverá HTML
                if "text/html" in r_file.headers.get("Content-Type", ""):
                    print(f"⚠️ El servidor rechazó la descarga para {nombre_fichero}. Saltando elemento...")
                    progreso.value = posicion
                    continue

                # Guardado progresivo en disco por bloques de 1MB para no saturar la RAM
                with open(nombre_fichero, "wb") as f_out:
                    for chunk in r_file.iter_content(chunk_size=1024 * 1024):
                        if chunk:
                            f_out.write(chunk)

            print(f"✅ ¡Guardado con éxito!")
            time.sleep(2.5)  # Pausa de cortesía prudencial para evitar saturar el servidor

        except requests.exceptions.RequestException as e:
            print(f"❌ Error de red al descargar {nombre_fichero}: {e}")
            time.sleep(5)

        # Actualizamos la barra de progreso
        progreso.value = posicion

    print("\n🎉 Proceso completado. Ficheros LiDAR descargados.")

except requests.exceptions.RequestException as e:
    print(f"❌ No se pudo inicializar la sesión con el buscador: {e}")


🔑 Registrando sesión legítima en el servidor mediante simulación de búsqueda...
✅ Sesión registrada correctamente en el CNIG. Iniciando descargas del GeoDataFrame...



IntProgress(value=0, description='Descargando:', max=20)

[1/20] Descargando: PNOA-2024-NAV-637-4717-H30-NPC03.laz (ID: 12617055)
⚠️ El servidor rechazó la descarga para PNOA-2024-NAV-637-4717-H30-NPC03.laz. Saltando elemento...
[2/20] Descargando: PNOA-2024-NAV-637-4718-H30-NPC03.laz (ID: 12614081)
⚠️ El servidor rechazó la descarga para PNOA-2024-NAV-637-4718-H30-NPC03.laz. Saltando elemento...
[3/20] Descargando: PNOA-2024-NAV-637-4719-H30-NPC03.laz (ID: 12613784)
⚠️ El servidor rechazó la descarga para PNOA-2024-NAV-637-4719-H30-NPC03.laz. Saltando elemento...
[4/20] Descargando: PNOA-2024-NAV-637-4720-H30-NPC03.laz (ID: 12612203)
⚠️ El servidor rechazó la descarga para PNOA-2024-NAV-637-4720-H30-NPC03.laz. Saltando elemento...
[5/20] Descargando: PNOA-2024-NAV-638-4717-H30-NPC03.laz (ID: 12613708)
⚠️ El servidor rechazó la descarga para PNOA-2024-NAV-638-4717-H30-NPC03.laz. Saltando elemento...
[6/20] Descargando: PNOA-2024-NAV-638-4718-H30-NPC03.laz (ID: 12610754)
⚠️ El servidor rechazó la descarga para PNOA-2024-NAV-638-4718-H30-NPC03.

In [19]:
# Generar un script de descargas automático para tu terminal de Windows o Linux
with open("descargar_lidar.sh", "w", encoding="utf-8") as f:
    f.write("#!/bin/bash\n")
    for i, row in lidar.iterrows():
        # Comando para guardar cada archivo manteniendo un retraso
        f.write(f'curl -L "https://centrodedescargas.cnig.es/CentroDescargas/descargaDir" --data "secuencialDescDir={row["num"]}" -o "archivo_{row["nombre"]}"\n')
        f.write("sleep 3\n")

print("Archivo 'descargar_lidar.sh' generado. Ejecútalo directamente en tu consola.")


Archivo 'descargar_lidar.sh' generado. Ejecútalo directamente en tu consola.


# PROCESAMIENTO LIDAR

In [1]:
def get_bounds(x, y):
    minx = int(x) * 1000
    maxx = minx + 1000
    maxy = int(y) * 1000
    miny = maxy - 1000

    return ([minx, maxx], [miny, maxy])

In [10]:
# import pdal
import os

municipio_name = 'SANGUESA'

os.makedirs(f'BRUTOS/{municipio_name}/LIDAR_FILES/', exist_ok=True)
os.makedirs(f'FINALES/{municipio_name}/LIDAR/ARBOLADO', exist_ok=True)
os.makedirs(f'FINALES/{municipio_name}/LIDAR/MATORRAL', exist_ok=True)

def get_arbolado(filename, x, y):
    vegetacion_pipeline = f"""{{
      "pipeline": [
         "/BRUTOS/{municipio_name}/LIDAR_FILES/{filename}",
        {{
            "type":"filters.hag_delaunay"
        }},
        {{
          "type": "filters.range",
          "limits": "HeightAboveGround[4:],Classification[3:5]"
        }},
        {{
          "type":"writers.gdal",
          "filename":"FINALES/{municipio_name}/ARBOLADO/arbolado_{x}_{y}.tif",
          "gdaldriver":"GTiff",
          "output_type":"max",
          "resolution":"5.0",
          "dimension":"HeightAboveGround",
          "bounds": "{get_bounds(x, y)}"
        }}
      ]
    }}"""

    pipeline = pdal.Pipeline(vegetacion_pipeline)
    pipeline.execute()

In [12]:
def get_matorral(filename, x, y):
    vegetacion_pipeline = f"""{{
      "pipeline": [
         "/BRUTOS/{municipio_name}/LIDAR_FILES/{filename}",
        {{
            "type":"filters.hag_delaunay"
        }},
        {{
          "type": "filters.range",
          "limits": "HeightAboveGround[:4],Classification[3:5]"
        }},
        {{
          "type":"writers.gdal",
          "filename":"FINALES/{municipio_name}/MATORRAL/matorral.tif",
          "gdaldriver":"GTiff",
          "output_type":"max",
          "resolution":"5.0",
          "dimension":"HeightAboveGround",
          "bounds": "{get_bounds(x, y)}"
        }}
      ]
    }}"""
    
    pipeline = pdal.Pipeline(vegetacion_pipeline)
    pipeline.execute()

In [8]:
from pathlib import Path
import contextlib
import rasterio
from rasterio.merge import merge

def merge_geotiffs(input_folder: str, output_filepath: str) -> None:
    """Fusiona todos los archivos GeoTIFF de una carpeta en un único archivo.
    
    :param input_folder: Ruta a la carpeta que contiene los ficheros .tif/.tiff
    :param output_filepath: Ruta y nombre del fichero GeoTIFF resultante
    """
    in_dir = Path(input_folder)
    out_file = Path(output_filepath)
    
    # 1. Buscar todos los archivos de imagen válidos
    tiff_files = list(in_dir.glob("*.tif")) + list(in_dir.glob("*.tiff"))
    
    if not tiff_files:
        raise FileNotFoundError(f"No se encontraron archivos GeoTIFF en la carpeta: {in_dir.resolve()}")
        
    print(f"Encontrados {len(tiff_files)} archivos para fusionar.")
    
    # 2. Abrir todos los archivos de manera segura usando ExitStack
    with contextlib.ExitStack() as stack:
        src_files = [stack.enter_context(rasterio.open(f)) for f in tiff_files]
        
        print("Calculando el mosaico y uniendo las imágenes...")
        # 3. Combinar las imágenes
        mosaic, out_transform = merge(src_files)
        
        # 4. Configurar metadatos del archivo de salida
        out_meta = src_files[0].meta.copy()
        out_meta.update({
            "driver": "GTiff",
            "height": mosaic.shape[1],  # Alto (filas)
            "width": mosaic.shape[2],   # Ancho (columnas)
            "transform": out_transform,
            "compress": "lzw"           # Compresión sin pérdida para optimizar espacio
        })
        
        # 5. Asegurar que el directorio de salida existe y guardar el archivo
        out_file.parent.mkdir(parents=True, exist_ok=True)
        with rasterio.open(out_file, "w", **out_meta) as dest:
            dest.write(mosaic)
            
    print(f"¡Proceso completado! Archivo guardado con éxito en: {out_file.resolve()}")

# --- Ejemplo de uso ---
# if __name__ == "__main__":
#     merge_geotiffs(
#         input_folder="./datos/canarias_lidar",
#         output_filepath="./resultados/mosaico_final.tif"
#     )


## Buena

In [2]:
municipio_name = 'SANGUESA'

In [3]:
import json
import numpy as np
import pandas as pd
import pdal
import rasterio
from rasterio.transform import from_origin
import os

os.makedirs(f'FINALES/{municipio_name}/FCC', exist_ok=True)
os.makedirs(f'FINALES/{municipio_name}/CBD', exist_ok=True)
os.makedirs(f'FINALES/{municipio_name}/MAT', exist_ok=True)
os.makedirs(f'FINALES/{municipio_name}/CONTINUIDAD_VERTICAL', exist_ok=True)

# ==============================================================================
# 1. PARÁMETROS DE CONFIGURACIÓN
# ==============================================================================
ARCHIVO_LAS = "tu_archivo.las"
TAMANO_PIXEL = 5.0     # Resolución espacial del TIFF (5x5 metros)
UMBRAL_DOSEL = 2.0     # Altura mínima para considerar dosel (metros)
PASO_VERTICAL = 2.0    # Grosor del estrato para el cálculo de CBD (metros)
EPSG_CODIGO = 25830    # Cambia esto por el EPSG de tu zona (ej: UTM 30N)
ALTURA_MAX_MAT = 2.0
# ==============================================================================
# 2. PIPELINE DE PDAL: NORMALIZACIÓN DE ALTURAS
# ==============================================================================
def get_CBD(filename, x, y):
    pipeline_json = f"""
    [
        "{filename}",
        {{
            "type":"filters.hag_delaunay"
        }},
        {{
          "type": "filters.range",
          "limits": "Classification[2:5]"
        }},
        {{
            "type": "filters.ferry",
            "dimensions": "HeightAboveGround=>Z"
        }}
    ]
    """
    
    print("Procesando la nube de puntos con PDAL...")
    pipeline = pdal.Pipeline(pipeline_json)
    pipeline.execute()
    
    # Extraer los datos normalizados
    data = pipeline.arrays[0]
    df = pd.DataFrame({
        'X': data['X'],
        'Y': data['Y'],
        'HAG': data['Z'],  # Altura sobre el suelo
        'Class': data['Classification']
    })
    
    # ==============================================================================
    # 3. CREACIÓN DE LA MATRIZ DEL RÁSTER
    # ==============================================================================
    # Determinar los límites espaciales del área de estudio
    range_x, range_y = get_bounds(x, y)
    x_min, x_max = range_x
    y_min, y_max = range_y

    print(range_x, range_y)
    
    # Ajustar las coordenadas para alinearlas al tamaño del píxel
    x_min_grid = x_min
    y_max_grid = y_max
    
    # Calcular las dimensiones de las imágenes ráster
    columnas = int(1000 / TAMANO_PIXEL)
    filas = int(1000 / TAMANO_PIXEL)
    
    # Crear matrices vacías rellenas con NoData (NaN)
    raster_fcc = np.full((filas, columnas), np.nan, dtype=np.float32)
    raster_cbd = np.full((filas, columnas), np.nan, dtype=np.float32)
    raster_mat = np.full((filas, columnas), np.nan, dtype=np.float32)
    raster_cont = np.full((filas, columnas), np.nan, dtype=np.float32)
    
    # Asignar índices de fila y columna a cada punto LiDAR
    df['col'] = ((df['X'] - x_min_grid) // TAMANO_PIXEL).astype(int)
    df['row'] = ((y_max_grid - df['Y']) // TAMANO_PIXEL).astype(int)
    
    # Asegurar que ningún índice se salga de la matriz por redondeos numéricos
    df = df[(df['col'] >= 0) & (df['col'] < columnas) & (df['row'] >= 0) & (df['row'] < filas)]
    
    print("Calculando métricas espaciales e indexando matriz...")
    
    # ==============================================================================
    # 4. PROCESAMIENTO PÍXEL A PÍXEL
    # ==============================================================================
    # Agrupamos los puntos por su píxel correspondiente
    for (f, c), grupo in df.groupby(['row', 'col']):
        alturas = grupo['HAG'].values
        total_puntos = len(alturas)
        
        if total_puntos == 0:
            continue
            
        # --- Cálculo de FCC ---
        puntos_dosel = np.sum(alturas > UMBRAL_DOSEL)
        fcc_valor = (puntos_dosel / total_puntos) * 100
        raster_fcc[f, c] = fcc_valor
        
        # --- Cálculo de CBD Proxy ---
        alturas_dosel = alturas[alturas > UMBRAL_DOSEL]
        if len(alturas_dosel) > 0:
            max_altura = alturas_dosel.max()
            bins = np.arange(UMBRAL_DOSEL, max_altura + PASO_VERTICAL, PASO_VERTICAL)
            
            if len(bins) >= 2:
                conteos, _ = np.histogram(alturas_dosel, bins=bins)
                volumen_estrato = (TAMANO_PIXEL ** 2) * PASO_VERTICAL
                densidades_volumetricas = conteos / volumen_estrato
                raster_cbd[f, c] = np.max(densidades_volumetricas)
            else:
                raster_cbd[f, c] = 0.0
        else:
            raster_cbd[f, c] = 0.0

        # --- Cálculo de Altura de matorral ---
        grupo_matorral = grupo[grupo['Class'].isin([3, 4, 5]) & (grupo['HAG'] < ALTURA_MAX_MAT) & (grupo['HAG'] > 0.15)]
        raster_mat[f, c] = grupo_matorral['HAG'].max()

        # --- Cáculo continuidad vertical ---
        puntos_vegetacion = grupo[grupo['Class'].isin([3, 4, 5]) & (grupo['HAG'] > 0.15)]
        if len(alturas_dosel) > 0:
            raster_cont[f, c] = len(grupo_matorral) / len(puntos_vegetacion)
        else:
            continue
    
    # ==============================================================================
    # 5. EXPORTACIÓN A GEOTIFF CON RASTERIO
    # ==============================================================================
    # Definir la geotransformación afín del ráster (origen y resolución)
    transformacion = from_origin(x_min_grid, y_max_grid, TAMANO_PIXEL, TAMANO_PIXEL)
    
    # Configuración de los metadatos comunes del archivo de salida
    metadatos = {
        'driver': 'GTiff',
        'height': filas,
        'width': columnas,
        'count': 1,
        'dtype': 'float32',
        'crs': rasterio.crs.CRS.from_epsg(EPSG_CODIGO),
        'transform': transformacion,
        'nodata': -9999.0
    }
    
    # Reemplazar valores NaN con el valor oficial de NoData para GIS
    raster_fcc = np.where(np.isnan(raster_fcc), -9999.0, raster_fcc)
    raster_cbd = np.where(np.isnan(raster_cbd), -9999.0, raster_cbd)
    raster_mat = np.where(np.isnan(raster_mat), -9999.0, raster_mat)
    raster_cont = np.where(np.isnan(raster_cont), -9999.0, raster_cont)
    
    # Guardar Ráster FCC
    with rasterio.open(f"FINALES/{municipio_name}/FCC/fcc_espesura_horizontal_{x}_{y}.tif", "w", **metadatos) as dst:
        dst.write(raster_fcc.astype(np.float32), 1)
    
    # Guardar Ráster CBD
    with rasterio.open(f"FINALES/{municipio_name}/CBD/cbd_espesura_volumetrica_{x}_{y}.tif", "w", **metadatos) as dst:
        dst.write(raster_cbd.astype(np.float32), 1)

    # Guardar Ráster MAT
    with rasterio.open(f"FINALES/{municipio_name}/MAT/matorral_{x}_{y}.tif", "w", **metadatos) as dst:
        dst.write(raster_mat.astype(np.float32), 1)

    # Guardar Ráster MAT
    with rasterio.open(f"FINALES/{municipio_name}/CONTINUIDAD_VERTICAL/continuidad_vertical_{x}_{y}.tif", "w", **metadatos) as dst:
        dst.write(raster_cont.astype(np.float32), 1)

    print("\n¡Proceso completado exitosamente!")
    print("-> Ráster FCC guardado en: fcc_espesura_horizontal.tif")
    print("-> Ráster CBD guardado en: cbd_espesura_volumetrica.tif")
    return data

In [4]:
import glob

a = glob.glob(f'BRUTOS/LIDAR/{municipio_name}/PNOA*')

In [5]:
for filename in a:
    # print(filename)
    coords = filename.split('/')[-1].split('_')[3]
    x, y = coords.split('-')
    if os.path.exists(f'FINALES/{municipio_name}/MAT/matorral_{x}_{y}.tif'):
        pass
    else:
        get_CBD(filename, x, y)

Procesando la nube de puntos con PDAL...
[640000, 641000] [4707000, 4708000]
Calculando métricas espaciales e indexando matriz...

¡Proceso completado exitosamente!
-> Ráster FCC guardado en: fcc_espesura_horizontal.tif
-> Ráster CBD guardado en: cbd_espesura_volumetrica.tif


In [9]:
merge_geotiffs(f'FINALES/{municipio_name}/CBD/', f'FINALES/{municipio_name}/merged_cbd.tif')

Encontrados 96 archivos para fusionar.
Calculando el mosaico y uniendo las imágenes...
¡Proceso completado! Archivo guardado con éxito en: /home/nano/anexos/FINALES/SANGUESA/merged_cbd.tif


In [10]:
merge_geotiffs(f'FINALES/{municipio_name}/FCC/', f'FINALES/{municipio_name}/merged_fcc.tif')

Encontrados 96 archivos para fusionar.
Calculando el mosaico y uniendo las imágenes...
¡Proceso completado! Archivo guardado con éxito en: /home/nano/anexos/FINALES/SANGUESA/merged_fcc.tif


In [11]:
merge_geotiffs(f'FINALES/{municipio_name}/CONTINUIDAD_VERTICAL/', f'FINALES/{municipio_name}/merged_continuidad_vertical.tif')

Encontrados 96 archivos para fusionar.
Calculando el mosaico y uniendo las imágenes...
¡Proceso completado! Archivo guardado con éxito en: /home/nano/anexos/FINALES/SANGUESA/merged_continuidad_vertical.tif


In [12]:
merge_geotiffs(f'FINALES/{municipio_name}/MAT/', f'FINALES/{municipio_name}/merged_matorral.tif')

Encontrados 96 archivos para fusionar.
Calculando el mosaico y uniendo las imágenes...
¡Proceso completado! Archivo guardado con éxito en: /home/nano/anexos/FINALES/SANGUESA/merged_matorral.tif
